In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
import string
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

print("? All libraries imported successfully!")

In [ ]:
df = pd.read_csv('../data/WELFake_Dataset.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nPercentage missing:")
print((df.isnull().sum() / len(df)) * 100)

In [ ]:
# Drop rows where both title and text are missing
df = df.dropna(subset=['text'])

# Fill missing titles with empty string
df['title'] = df['title'].fillna('')

# Combine title and text into one column
df['content'] = df['title'] + ' ' + df['text']

print("? Missing values handled!")
print("New shape:", df.shape)

In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

df['cleaned_text'] = df['content'].apply(clean_text)

print("? Text cleaned!")
print("\nBefore cleaning:")
print(df['content'][0][:200])
print("\nAfter cleaning:")
print(df['cleaned_text'][0][:200])

In [ ]:
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens

df['tokens'] = df['cleaned_text'].apply(tokenize_text)

print("? Tokenization done!")
print("\nSample tokens:")
print(df['tokens'][0][:20])

In [ ]:
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens_no_stop'] = df['tokens'].apply(remove_stopwords)

print("? Stopwords removed!")
print("\nBefore:", df['tokens'][0][:15])
print("After:", df['tokens_no_stop'][0][:15])

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['tokens_lemmatized'] = df['tokens_no_stop'].apply(lemmatize_tokens)
df['final_text'] = df['tokens_lemmatized'].apply(lambda x: ' '.join(x))

print("? Lemmatization done!")
print("\nSample final text:")
print(df['final_text'][0][:200])

In [ ]:
df[['final_text', 'label']].to_csv('../data/welFake_cleaned.csv', index=False)

print("? Cleaned dataset saved!")
print("Shape of cleaned data:", df.shape)
print("\nLabel distribution:")
print(df['label'].value_counts())